# Tiered Prefix Cache: CPU and Disk Offloading

GPU HBM (High Bandwidth Memory) is the fastest but most expensive memory
tier. On an H100, you get 80 GB of HBM shared between model weights and
the KV-cache. When the KV-cache fills up, vLLM evicts old entries, and
future requests for those prefixes pay the full prefill cost again.

**Tiered prefix caching** extends the KV-cache across three memory tiers:

1. **GPU HBM** - Fastest (3.35 TB/s on H100). Active cache entries live here.
2. **CPU DRAM** - Large and affordable (~200 GB/s). Evicted entries move here
   instead of being discarded.
3. **Disk (NVMe SSD)** - Largest and cheapest (~7 GB/s). Overflow from CPU DRAM.

This hierarchy means evicted KV-cache blocks are demoted rather than deleted.
When a request arrives that matches a demoted prefix, the blocks are promoted
back to GPU HBM, which is much faster than recomputing the prefill.

By the end of this notebook, you will:
- Deploy the tiered prefix cache guide with CPU offloading
- Configure vLLM's cpu-offload-gb parameter
- Configure the precise-prefix-cache-scorer with tiered awareness
- Run a high-concurrency workload and monitor tier distribution
- Compare throughput against GPU-only caching

In [ ]:
# Environment setup
import os

os.environ["NAMESPACE"] = "llm-d"
os.environ["MODEL_NAME"] = "Qwen/Qwen3-32B"

print("Namespace:", os.environ["NAMESPACE"])
print("Model:", os.environ["MODEL_NAME"])

## How Tiered Caching Works

### Without Tiered Caching

```
GPU HBM (80 GB)
+-----------------+
| Model Weights   |  ~60 GB for a 32B model (FP16)
+-----------------+
| KV-Cache        |  ~20 GB available
| [block A]       |  When full, oldest blocks are DELETED
| [block B]       |
| [block C]       |
+-----------------+
```

### With Tiered Caching

```
GPU HBM (80 GB)          CPU DRAM (128 GB)       NVMe SSD (1 TB)
+-----------------+      +-----------------+     +-----------------+
| Model Weights   |      | Demoted blocks  |     | Overflow blocks |
+-----------------+      | [block D]       |     | [block G]       |
| KV-Cache (hot)  |      | [block E]       |     | [block H]       |
| [block A]       | ---> | [block F]       | --> | [block I]       |
| [block B]       |evict |                 |evict|                 |
| [block C]       |      |                 |     |                 |
+-----------------+      +-----------------+     +-----------------+
         ^                        |
         |     promote (fast)     |
         +------------------------+
```

Promotion from CPU DRAM to GPU HBM takes ~0.5ms for a typical cache block,
compared to 10-50ms for recomputing the prefill from scratch. Promotion
from NVMe takes ~2-5ms, still much faster than recomputation.

In [ ]:
# Deploy the tiered prefix cache guide
# This configures vLLM with CPU offloading and the EPP with tier-aware scoring

!kubectl apply -k llm-d-production-stack/guides/tiered-prefix-cache/model-server/ \
    -n $NAMESPACE

print("Model server deployed with CPU offload connector.")
print("Waiting for pods (this may take longer due to additional memory allocation)...")
!kubectl wait --for=condition=ready pod -l app=vllm -n $NAMESPACE --timeout=600s
print("Model server pods are ready.")

In [ ]:
# Inspect the vLLM configuration to see the CPU offload settings

print("=== vLLM CPU Offload Configuration ===")
print()
!kubectl get deployment vllm -n $NAMESPACE -o yaml | grep -A 5 'cpu-offload'

print()
print("Key parameters:")
print("  --cpu-offload-gb 64")
print("    Allocates 64 GB of CPU DRAM for KV-cache offloading.")
print("    This is in addition to the GPU HBM cache.")
print()
print("  The node must have sufficient free DRAM. Check node resources:")
!kubectl describe node $(kubectl get pods -n llm-d -l app=vllm -o jsonpath='{.items[0].spec.nodeName}') \
    | grep -A 5 'Allocatable:' | head -6

In [ ]:
# Deploy the router with tier-aware precise prefix cache scoring

!kubectl apply -k llm-d-production-stack/guides/tiered-prefix-cache/router/ \
    -n $NAMESPACE

print("Router deployed with tier-aware cache scoring.")
!kubectl wait --for=condition=ready pod -l app=epp -n $NAMESPACE --timeout=120s
print("EPP pods are ready.")

print()
print("The precise-prefix-cache-scorer now accounts for blocks in all tiers:")
print("  GPU HBM:  weight 1.0 (instant access, highest score)")
print("  CPU DRAM: weight 0.8 (fast promotion, still valuable)")
print("  NVMe:     weight 0.3 (slower promotion, but better than recompute)")
print()
print("The scorer prefers replicas with GPU-cached blocks, but will also")
print("route to replicas with CPU or disk-cached blocks over replicas")
print("with no cache match at all.")

In [ ]:
# Run a high-concurrency workload to exercise tiered caching
# 250 concurrent users with a mix of prompts will fill the GPU cache
# and cause evictions to CPU DRAM

import subprocess
import json
import time
import threading
import random

GATEWAY_IP = subprocess.check_output(
    "kubectl get svc envoy-gateway -n llm-d "
    "-o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

# Generate a large set of prompts to fill the cache
# Some prompts share prefixes (simulating RAG with shared document context)
shared_prefix = (
    "You are an expert assistant. The following document describes the "
    "architecture of a modern cloud-native application. The application "
    "uses microservices, event-driven communication, and container "
    "orchestration. Each service is independently deployable and scalable. "
)

questions = [
    "What are the benefits of microservices?",
    "How does event-driven communication work?",
    "What is container orchestration?",
    "How do you scale individual services?",
    "What are the challenges of distributed systems?",
    "How do you handle service discovery?",
    "What is a service mesh?",
    "How do you implement circuit breakers?",
    "What is eventual consistency?",
    "How do you monitor microservices?",
]

stats = {"sent": 0, "completed": 0, "errors": 0}
stop_event = threading.Event()

def send_requests_worker():
    while not stop_event.is_set():
        question = random.choice(questions)
        prompt = shared_prefix + question
        payload = json.dumps({
            "model": "Qwen/Qwen3-32B",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 100,
        })
        stats["sent"] += 1
        result = subprocess.run(
            ["curl", "-s", "-m", "30",
             f"http://{GATEWAY_IP}:8080/v1/chat/completions",
             "-H", "Content-Type: application/json", "-d", payload],
            capture_output=True, text=True
        )
        if result.returncode == 0 and '"error"' not in result.stdout:
            stats["completed"] += 1
        else:
            stats["errors"] += 1

# Launch 250 concurrent workers
print(f"Launching 250 concurrent workers against {GATEWAY_IP}:8080...")
print("This will fill the GPU KV-cache and trigger evictions to CPU DRAM.")
print()

threads = [threading.Thread(target=send_requests_worker, daemon=True)
           for _ in range(250)]
for t in threads:
    t.start()

# Monitor for 3 minutes
for i in range(12):
    time.sleep(15)
    print(f"  [{(i+1)*15:>3}s] Sent: {stats['sent']:>5}, "
          f"Completed: {stats['completed']:>5}, "
          f"Errors: {stats['errors']:>3}")

stop_event.set()
time.sleep(2)
print(f"\nWorkload complete. Total sent: {stats['sent']}, "
      f"Completed: {stats['completed']}, Errors: {stats['errors']}")

In [ ]:
# Monitor cache tier distribution
# See how entries are distributed across GPU, CPU, and disk tiers

print("=== Cache Tier Distribution ===")
print()

# Query tier-specific metrics
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=vllm_cache_blocks_total{namespace="llm-d"}' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    tier = r['metric'].get('tier', 'unknown')
    pod = r['metric'].get('pod', 'unknown')
    blocks = int(float(r['value'][1]))
    print(f'  {pod} [{tier}]: {blocks} blocks')
" 2>/dev/null || echo "  (tier metrics not yet available)"

print()

# Cache utilization per tier
print("Cache Utilization:")
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=vllm_gpu_cache_usage_perc{namespace="llm-d"}' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    pod = r['metric'].get('pod', 'unknown')
    pct = float(r['value'][1]) * 100
    print(f'  {pod} GPU cache: {pct:.1f}% full')
" 2>/dev/null || echo "  (utilization metrics not yet available)"

print()
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=vllm_cpu_cache_usage_perc{namespace="llm-d"}' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    pod = r['metric'].get('pod', 'unknown')
    pct = float(r['value'][1]) * 100
    print(f'  {pod} CPU cache: {pct:.1f}% full')
" 2>/dev/null || echo "  (CPU cache metrics not yet available)"

print()
print("Expected distribution after high-concurrency workload:")
print("  GPU HBM:  90-100% full (hot entries)")
print("  CPU DRAM: 30-60% full (recently evicted entries)")
print("  NVMe:     0-20% full (oldest evicted entries, if configured)")

In [ ]:
# Compare throughput: tiered cache vs. GPU-only
# With tiered caching, more cache hits means less recomputation,
# which translates directly to higher throughput

print("=== Throughput Comparison ===")
print()

# Query current throughput
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=sum(rate(epp_requests_total{namespace="llm-d",status="success"}[5m]))' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    rps = float(r['value'][1])
    print(f'  Current throughput: {rps:.1f} req/s (tiered caching enabled)')
" 2>/dev/null || echo "  (throughput metrics not yet available)"

print()

# Cache hit rate (higher with tiered caching)
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=avg(epp_prefix_cache_hit_ratio{namespace="llm-d"})' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    val = float(r['value'][1])
    print(f'  Cache hit rate: {val:.2%}')
" 2>/dev/null || echo "  (cache metrics not yet available)"

print()
print("Benchmark results from the llm-d team (ShareGPT workload):")
print()
print("  Configuration          | Throughput | Cache Hit Rate | TTFT p50")
print("  ---------------------- | ---------- | -------------- | --------")
print("  GPU-only (20 GB cache) |   1.0x     |     35%        |  120ms")
print("  GPU + CPU (84 GB)      |   5.2x     |     72%        |   45ms")
print("  GPU + CPU + NVMe       |  13.9x     |     91%        |   28ms")
print()
print("The improvement scales with workload locality. Workloads with")
print("high prefix reuse (multi-turn chat, RAG) benefit the most.")

## Summary

Tiered prefix caching extends the KV-cache beyond GPU HBM into CPU DRAM
and NVMe SSD. This dramatically increases effective cache capacity and
reduces cache eviction pressure under high concurrency.

### Key Configuration

- **vLLM**: `--cpu-offload-gb 64` to allocate CPU DRAM for cache offloading
- **EPP**: Precise-prefix-cache-scorer configured with tier weights
  (GPU=1.0, CPU=0.8, NVMe=0.3)
- **Deploy**: `guides/tiered-prefix-cache` in the production stack

### Performance Impact

- Up to **13.9x throughput improvement** with GPU + CPU + NVMe tiers
- Cache hit rates increase from ~35% (GPU-only) to ~91% (all tiers)
- TTFT p50 drops significantly due to more cache hits
- CPU DRAM promotion latency (~0.5ms) is negligible compared to
  recomputation (10-50ms)

### Hardware Requirements

- Sufficient CPU DRAM on GPU nodes (check with `kubectl describe node`)
- NVMe SSDs if using the disk tier (optional)
- No additional GPUs required

### Next Steps

- **Wide Expert-Parallelism**: For MoE models, wide EP distributes
  experts across nodes and also increases per-node KV-cache space.
- **Precise Prefix Cache**: Combine tiered caching with precise
  event-driven cache tracking for maximum routing accuracy.